# SolarSDE Notebook 4 — Ablation Study

**Runtime:** ~3-5 hours on Colab T4 / Kaggle P100

**Prerequisite:** Notebooks 1 and 2 (needs latents + main results).

**Ablations:**
- **A2:** SolarSDE minus CTI gating (diffusion uses constant CTI = 0 ⇒ state-independent)
- **A4:** SolarSDE minus Score Matching (linear decoder z → GHI with Gaussian noise)
- **A5:** SolarSDE minus Neural SDE (deterministic ODE: σ_θ ≡ 0)

Each ablation re-trains the relevant component(s) and evaluates at the same 5 horizons as Notebook 2.


In [ ]:
# ==== Install dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("pvlib", "properscoring", "pyarrow", "tqdm")
print("Dependencies installed.")


In [ ]:
# ==== Environment Detection & Setup ====
import os, sys, time, json, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

PERSIST_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PERSIST_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LATENT_DIR = PERSIST_DIR / "latents"
LATENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")
print(f"Working directory:  {WORK_DIR}")


In [ ]:
# ==== GPU Check ====
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:          {torch.cuda.get_device_name(0)}")
    print(f"Memory:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: Running on CPU — will be slow. Change Runtime > Change runtime type > GPU")
print(f"Using device: {DEVICE}")


## Fast-start — pull Notebook 1 outputs from GitHub (skips VAE retraining)

In [ ]:
# ==== Fast-start: fetch Notebook 1 outputs from GitHub if persistent storage is empty ====
# This lets Notebooks 2-5 run standalone without having re-executed Notebook 1.
GITHUB_REPO = "https://github.com/keshavkrishnan08/SDE"
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

need_nb1_outputs = not (CHECKPOINT_DIR / "vae_best.pt").exists() or not any(LATENT_DIR.glob("train_*.npy"))
if need_nb1_outputs:
    print("Notebook 1 outputs not found in persistent storage — pulling from GitHub ...")
    import requests
    files_to_pull = [
        ("checkpoints/vae_best.pt",          CHECKPOINT_DIR / "vae_best.pt"),
        ("checkpoints/vae_final.pt",         CHECKPOINT_DIR / "vae_final.pt"),
        ("results/vae_training_history.csv", RESULTS_DIR / "vae_training_history.csv"),
        ("splits/train.parquet",             PERSIST_DIR / "splits" / "train.parquet"),
        ("splits/val.parquet",               PERSIST_DIR / "splits" / "val.parquet"),
        ("splits/test.parquet",              PERSIST_DIR / "splits" / "test.parquet"),
    ]
    for split in ["train", "val", "test"]:
        for k in ["latents", "cti", "ghi", "covariates", "is_ramp"]:
            files_to_pull.append((f"latents/{split}_{k}.npy", LATENT_DIR / f"{split}_{k}.npy"))

    for rel, dest in files_to_pull:
        url = f"{GITHUB_RAW}/colab_outputs/{rel}"
        if dest.exists() and dest.stat().st_size > 0:
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(url, timeout=120)
            if r.status_code == 200 and len(r.content) > 100:
                dest.write_bytes(r.content)
                print(f"  OK  {rel}  ({len(r.content)/1e6:.2f} MB)")
            else:
                print(f"  SKIP {rel}  (status {r.status_code})")
        except Exception as e:
            print(f"  FAIL {rel}: {e}")
    print("Fast-start pull complete.")
else:
    print("Notebook 1 outputs already present in persistent storage.")


In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import time

def load_split(s):
    return {k: np.load(LATENT_DIR / f"{s}_{k}.npy") for k in ["latents", "cti", "ghi", "covariates", "is_ramp"]}

data = {s: load_split(s) for s in ["train", "val", "test"]}
# Re-key for convenience
for s in data:
    data[s] = {"Z": data[s]["latents"], "cti": data[s]["cti"], "ghi": data[s]["ghi"],
               "cov": data[s]["covariates"], "ramp": data[s]["is_ramp"]}

Z_DIM = data["train"]["Z"].shape[1]
C_DIM = max(1, data["train"]["cov"].shape[1])
HORIZONS = [6, 30, 60, 120, 180]
HORIZON_MIN = {6: 1, 30: 5, 60: 10, 120: 20, 180: 30}
N_SAMPLES = 50
N_EVAL = min(1000, len(data["test"]["Z"]) - max(HORIZONS) - 1)
print(f"Z_DIM={Z_DIM}, C_DIM={C_DIM}, N_EVAL={N_EVAL}")


## Shared model code

In [ ]:
# ==== Neural SDE model ====
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class DriftNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(inplace=True),
            nn.Linear(h, h), nn.SiLU(inplace=True),
            ResBlock(h), ResBlock(h),
            nn.Linear(h, z_dim),
        )
    def forward(self, z, t, c):
        return self.net(torch.cat([z, t, c], dim=-1))

class CTIDiffNet(nn.Module):
    """CTI-gated diffusion: σ = Softplus(MLP(z) * Softplus(MLP(CTI)))"""
    def __init__(self, z_dim=64, h=64):
        super().__init__()
        self.cti_gate = nn.Sequential(nn.Linear(1, h), nn.Softplus())
        self.state = nn.Sequential(nn.Linear(z_dim, h), nn.SiLU(inplace=True))
        self.out = nn.Sequential(nn.Linear(h, z_dim), nn.Softplus())
    def forward(self, z, cti):
        return self.out(self.state(z) * self.cti_gate(cti))

class LatentNeuralSDE(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, drift_h=256, diff_h=64, lambda_sigma=1.0):
        super().__init__()
        self.z_dim = z_dim
        self.lambda_sigma = lambda_sigma
        self.drift = DriftNet(z_dim, c_dim, drift_h)
        self.diffusion = CTIDiffNet(z_dim, diff_h)
    def forward(self, z, t, c, cti):
        return self.drift(z, t, c), self.diffusion(z, cti)
    def sde_matching_loss(self, z, zn, t, c, cti, dt=1.0):
        mu = self.drift(z, t, c)
        sigma = self.diffusion(z, cti)
        dz = (zn - z) / dt
        drift_l = F.mse_loss(mu, dz)
        resid = (zn - z - mu * dt).pow(2) / dt
        diff_l = F.mse_loss(sigma.pow(2), resid)
        return {"loss": drift_l + self.lambda_sigma * diff_l,
                "drift": drift_l, "diffusion": diff_l}


In [ ]:
# ==== Conditional Score-Matching Irradiance Decoder (CSMID) ====
class ScoreRes(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class ScoreNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2):
        super().__init__()
        d_in = 1 + 1 + z_dim + 1 + c_dim
        layers = [nn.Linear(d_in, h), nn.SiLU(inplace=True)]
        for _ in range(blocks): layers.append(ScoreRes(h))
        layers.append(nn.Linear(h, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, g, s, z, cti, c):
        return self.net(torch.cat([g, s, z, cti, c], dim=-1))

class CondScoreDecoder(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2, steps=100, b0=1e-4, b1=0.02):
        super().__init__()
        self.steps = steps
        self.score = ScoreNet(z_dim, c_dim, h, blocks)
        betas = torch.linspace(b0, b1, steps)
        alphas = 1 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cum", ac)
        self.register_buffer("sac", torch.sqrt(ac))
        self.register_buffer("s1mac", torch.sqrt(1 - ac))
    def q(self, g0, si, eps):
        return self.sac[si].unsqueeze(-1) * g0 + self.s1mac[si].unsqueeze(-1) * eps
    def training_loss(self, g0, z, cti, c):
        B = g0.shape[0]
        si = torch.randint(0, self.steps, (B,), device=g0.device)
        sn = (si.float() / self.steps).unsqueeze(-1)
        eps = torch.randn_like(g0)
        gs = self.q(g0, si, eps)
        pred = self.score(gs, sn, z, cti, c)
        target = -eps / self.s1mac[si].unsqueeze(-1)
        return {"loss": F.mse_loss(pred, target)}
    @torch.no_grad()
    def sample(self, z, cti, c, n=1):
        B = z.shape[0]
        z_e = z.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        cti_e = cti.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        c_e = c.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        x = torch.randn(B * n, 1, device=z.device)
        for i in reversed(range(self.steps)):
            sn = torch.full((B * n, 1), i / self.steps, device=z.device)
            score = self.score(x, sn, z_e, cti_e, c_e)
            b, a, ac = self.betas[i], self.alphas[i], self.alphas_cum[i]
            mean = (1 / a.sqrt()) * (x + b * score * (1 - ac).sqrt())
            if i > 0:
                x = mean + b.sqrt() * torch.randn_like(x)
            else:
                x = mean
        return x.view(B, n)


In [ ]:
# ==== Probabilistic forecasting metrics ====
def crps_empirical(y_true, y_samples):
    """CRPS from empirical samples. y_true: (N,), y_samples: (N, M)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    y_med = np.median(y_samples, axis=1)
    y_range = y_true.max() - y_true.min()
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps": float(crps.mean()),
        "picp": picp(y_true, y_samples, alpha),
        "pinaw": pinaw(y_samples, y_range, alpha),
        "rmse": rmse(y_true, y_med),
        "mae":  mae(y_true, y_med),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    return out


In [ ]:
# ==== Euler-Maruyama SDE solver ====
def em_step(drift_fn, diff_fn, z, t, c, cti, dt):
    mu = drift_fn(z, t, c)
    sigma = diff_fn(z, cti)
    return z + mu * dt + sigma * (dt ** 0.5) * torch.randn_like(z)

def solve_sde_horizons(sde, z0, horizons, c, cti, N=50, dt=1.0):
    B, d = z0.shape
    mx = max(horizons)
    hset = set(horizons)
    z = z0.unsqueeze(1).expand(B, N, d).reshape(B * N, d)
    c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    out = {}
    for step in range(mx):
        t = torch.full((B * N, 1), float(step), device=z0.device)
        z = em_step(sde.drift, sde.diffusion, z, t, c_e, cti_e, dt)
        if (step + 1) in hset:
            out[step + 1] = z.view(B, N, d).clone()
    return out


In [ ]:
class LatentSeqDataset(Dataset):
    def __init__(self, d):
        self.Z=d["Z"]; self.cti=d["cti"]; self.ghi=d["ghi"]; self.cov=d["cov"]
    def __len__(self): return max(0, len(self.Z) - 1)
    def __getitem__(self, i):
        return {"z_t": torch.from_numpy(self.Z[i]).float(),
                "z_next": torch.from_numpy(self.Z[i+1]).float(),
                "cti": torch.tensor(float(self.cti[i])),
                "ghi": torch.tensor(float(self.ghi[i])),
                "cov": torch.from_numpy(self.cov[i]).float() if self.cov.shape[1] > 0 else torch.zeros(C_DIM)}

tr_ds = LatentSeqDataset(data["train"])
va_ds = LatentSeqDataset(data["val"])


## A2 — SolarSDE minus CTI gating

In [ ]:
# No-CTI variant: diffusion conditioned on constant zero CTI (ablates the gating mechanism)
print("=" * 70)
print("A2: SolarSDE without CTI gating")
print("=" * 70)
torch.manual_seed(42); np.random.seed(42)

sde_a2 = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM, drift_h=256, diff_h=64, lambda_sigma=1.0).to(DEVICE)
opt = torch.optim.Adam(sde_a2.parameters(), lr=1e-4)
dl = DataLoader(tr_ds, batch_size=128, shuffle=True, drop_last=True)
vl = DataLoader(va_ds, batch_size=128, shuffle=False)
EPOCHS = 100; best = float("inf"); t0 = time.time()
for ep in range(1, EPOCHS + 1):
    sde_a2.train(); tl = 0; n = 0
    for b in dl:
        z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
        cti0 = torch.zeros(z.shape[0], 1, device=DEVICE)  # ABLATION: constant CTI=0
        c = b["cov"].to(DEVICE); t = torch.zeros(z.shape[0], 1, device=DEVICE)
        l = sde_a2.sde_matching_loss(z, zn, t, c, cti0)["loss"]
        opt.zero_grad(); l.backward()
        torch.nn.utils.clip_grad_norm_(sde_a2.parameters(), 1.0); opt.step()
        tl += l.item(); n += 1
    sde_a2.eval(); vl_s = vn = 0
    with torch.no_grad():
        for b in vl:
            z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
            cti0 = torch.zeros(z.shape[0], 1, device=DEVICE)
            c = b["cov"].to(DEVICE); t = torch.zeros(z.shape[0], 1, device=DEVICE)
            vl_s += sde_a2.sde_matching_loss(z, zn, t, c, cti0)["loss"].item(); vn += 1
    vl_s /= max(vn, 1)
    if ep % 20 == 0: print(f"  A2 epoch {ep}: train={tl/n:.5f} val={vl_s:.5f} time={(time.time()-t0)/60:.1f}min")
    if vl_s < best: best = vl_s; torch.save(sde_a2.state_dict(), CHECKPOINT_DIR / "sde_a2_best.pt")

# Use original Score Decoder (from Notebook 2)
score = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, h=256, blocks=2, steps=100).to(DEVICE)
score.load_state_dict(torch.load(CHECKPOINT_DIR / "score_best.pt", map_location=DEVICE))
score.eval()
sde_a2.load_state_dict(torch.load(CHECKPOINT_DIR / "sde_a2_best.pt", map_location=DEVICE))
sde_a2.eval()

# Evaluate A2
te = data["test"]
res_a2 = {}
for h in HORIZONS:
    yt, ys, rm = [], [], []
    for i in range(0, N_EVAL, 32):
        idx = list(range(i, min(i + 32, N_EVAL)))
        z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
        c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
        cti0 = torch.zeros(len(idx), 1, device=DEVICE)  # forecast with CTI=0
        with torch.no_grad():
            endp = solve_sde_horizons(sde_a2, z0, [h], c, cti0, N=N_SAMPLES, dt=1.0)[h]
            B, N, d = endp.shape
            g = score.sample(endp.view(B*N, d),
                             cti0.unsqueeze(1).expand(B, N, -1).reshape(B*N, -1),
                             c.unsqueeze(1).expand(B, N, -1).reshape(B*N, -1), n=1).squeeze(-1).view(B, N).cpu().numpy()
        for k, ii in enumerate(idx):
            j = ii + h
            if j < len(te["ghi"]): yt.append(te["ghi"][j]); ys.append(g[k]); rm.append(te["ramp"][j])
    m = all_metrics(np.array(yt), np.array(ys), is_ramp=np.array(rm))
    m["horizon_min"] = HORIZON_MIN[h]; m["variant"] = "A2_no_cti"
    res_a2[h] = m
    print(f"  A2 h={HORIZON_MIN[h]}min: CRPS={m['crps']:.3f} PICP={m['picp']:.3f}")

df_a2 = pd.DataFrame.from_dict(res_a2, orient="index").sort_values("horizon_min")
df_a2.to_csv(RESULTS_DIR / "ablation_a2_no_cti.csv", index=False)


## A4 — SolarSDE minus Score Matching (linear decoder)

In [ ]:
# Train a linear z→GHI decoder with Gaussian residual noise for probabilistic output
print("=" * 70)
print("A4: SolarSDE without Score Matching (linear decoder)")
print("=" * 70)

class LinearDecoder(nn.Module):
    def __init__(self, z_dim, c_dim, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(), nn.Linear(h, 1))
    def forward(self, z, cti, c):
        return self.net(torch.cat([z, cti, c], dim=-1)).squeeze(-1)

torch.manual_seed(42)
lin = LinearDecoder(Z_DIM, C_DIM, 64).to(DEVICE)
opt = torch.optim.Adam(lin.parameters(), lr=1e-3); crit = nn.MSELoss()
dl = DataLoader(tr_ds, batch_size=256, shuffle=True, drop_last=True)

EPOCHS = 30
for ep in range(1, EPOCHS + 1):
    lin.train(); tl = 0; n = 0
    for b in dl:
        z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
        c = b["cov"].to(DEVICE); g = b["ghi"].to(DEVICE)
        loss = crit(lin(z, cti, c), g)
        opt.zero_grad(); loss.backward(); opt.step(); tl += loss.item(); n += 1
    if ep % 10 == 0: print(f"  A4 epoch {ep}: train={tl/n:.3f}")
torch.save(lin.state_dict(), CHECKPOINT_DIR / "linear_decoder_a4.pt")

# Calibrate Gaussian noise on train residuals
lin.eval()
with torch.no_grad():
    z_all = torch.from_numpy(data["train"]["Z"]).float().to(DEVICE)
    cti_all = torch.from_numpy(data["train"]["cti"]).float().unsqueeze(-1).to(DEVICE)
    c_all = torch.from_numpy(data["train"]["cov"]).float().to(DEVICE)
    pred = lin(z_all, cti_all, c_all).cpu().numpy()
g_true = data["train"]["ghi"]
a4_std = float(np.std(g_true - pred))
print(f"  Linear decoder residual std: {a4_std:.2f} W/m²")

# Use original SDE from Notebook 2 for latent propagation
sde_full = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM, drift_h=256, diff_h=64).to(DEVICE)
sde_full.load_state_dict(torch.load(CHECKPOINT_DIR / "sde_best.pt", map_location=DEVICE))
sde_full.eval()

rng = np.random.default_rng(42)
te = data["test"]
res_a4 = {}
for h in HORIZONS:
    yt, ys, rm = [], [], []
    for i in range(0, N_EVAL, 32):
        idx = list(range(i, min(i + 32, N_EVAL)))
        z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
        c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
        cti = torch.from_numpy(te["cti"][idx]).float().unsqueeze(-1).to(DEVICE)
        with torch.no_grad():
            endp = solve_sde_horizons(sde_full, z0, [h], c, cti, N=N_SAMPLES, dt=1.0)[h]
            B, N, d = endp.shape
            cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
            c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
            pred = lin(endp.view(B * N, d), cti_e, c_e).view(B, N).cpu().numpy()
            noise = rng.normal(0, a4_std, size=(B, N))
            g = np.clip(pred + noise, 0, None)
        for k, ii in enumerate(idx):
            j = ii + h
            if j < len(te["ghi"]): yt.append(te["ghi"][j]); ys.append(g[k]); rm.append(te["ramp"][j])
    m = all_metrics(np.array(yt), np.array(ys), is_ramp=np.array(rm))
    m["horizon_min"] = HORIZON_MIN[h]; m["variant"] = "A4_no_score"
    res_a4[h] = m
    print(f"  A4 h={HORIZON_MIN[h]}min: CRPS={m['crps']:.3f} PICP={m['picp']:.3f}")

df_a4 = pd.DataFrame.from_dict(res_a4, orient="index").sort_values("horizon_min")
df_a4.to_csv(RESULTS_DIR / "ablation_a4_no_score.csv", index=False)


## A5 — SolarSDE minus Neural SDE (deterministic ODE: σ ≡ 0)

In [ ]:
print("=" * 70)
print("A5: SolarSDE deterministic (Neural ODE, σ=0)")
print("=" * 70)

# Train drift-only; diffusion branch frozen to zero at inference
torch.manual_seed(42)
sde_a5 = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM, drift_h=256, diff_h=64, lambda_sigma=0.0).to(DEVICE)
opt = torch.optim.Adam(sde_a5.drift.parameters(), lr=1e-4)   # only train drift
dl = DataLoader(tr_ds, batch_size=128, shuffle=True, drop_last=True)

EPOCHS = 100
for ep in range(1, EPOCHS + 1):
    sde_a5.train(); tl = 0; n = 0
    for b in dl:
        z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
        c = b["cov"].to(DEVICE); t = torch.zeros(z.shape[0], 1, device=DEVICE)
        dz = (zn - z) / 1.0
        mu = sde_a5.drift(z, t, c)
        l = F.mse_loss(mu, dz)
        opt.zero_grad(); l.backward(); opt.step(); tl += l.item(); n += 1
    if ep % 20 == 0: print(f"  A5 epoch {ep}: drift_loss={tl/n:.5f}")
torch.save(sde_a5.state_dict(), CHECKPOINT_DIR / "sde_a5_best.pt")

# For deterministic forecast, all samples are identical → collapse. Use score decoder stochasticity.
score = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, h=256, blocks=2, steps=100).to(DEVICE)
score.load_state_dict(torch.load(CHECKPOINT_DIR / "score_best.pt", map_location=DEVICE))
score.eval(); sde_a5.eval()

def solve_ode_horizons(drift_fn, z0, horizons, c, dt=1.0):
    B, d = z0.shape
    mx = max(horizons); hset = set(horizons); out = {}
    z = z0.clone()
    for step in range(mx):
        t = torch.full((B, 1), float(step), device=z.device)
        z = z + drift_fn(z, t, c) * dt
        if (step + 1) in hset: out[step + 1] = z.clone()
    return out

te = data["test"]
res_a5 = {}
for h in HORIZONS:
    yt, ys, rm = [], [], []
    for i in range(0, N_EVAL, 32):
        idx = list(range(i, min(i + 32, N_EVAL)))
        z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
        c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
        cti = torch.from_numpy(te["cti"][idx]).float().unsqueeze(-1).to(DEVICE)
        with torch.no_grad():
            endp = solve_ode_horizons(sde_a5.drift, z0, [h], c, dt=1.0)[h]  # (B, d) deterministic
            # Duplicate endpoints N_SAMPLES times so score decoder provides stochasticity
            endp_rep = endp.unsqueeze(1).expand(-1, N_SAMPLES, -1).reshape(-1, Z_DIM)
            cti_rep = cti.unsqueeze(1).expand(-1, N_SAMPLES, -1).reshape(-1, 1)
            c_rep = c.unsqueeze(1).expand(-1, N_SAMPLES, -1).reshape(-1, C_DIM)
            g = score.sample(endp_rep, cti_rep, c_rep, n=1).squeeze(-1).view(len(idx), N_SAMPLES).cpu().numpy()
        for k, ii in enumerate(idx):
            j = ii + h
            if j < len(te["ghi"]): yt.append(te["ghi"][j]); ys.append(g[k]); rm.append(te["ramp"][j])
    m = all_metrics(np.array(yt), np.array(ys), is_ramp=np.array(rm))
    m["horizon_min"] = HORIZON_MIN[h]; m["variant"] = "A5_deterministic_ode"
    res_a5[h] = m
    print(f"  A5 h={HORIZON_MIN[h]}min: CRPS={m['crps']:.3f} PICP={m['picp']:.3f}")

df_a5 = pd.DataFrame.from_dict(res_a5, orient="index").sort_values("horizon_min")
df_a5.to_csv(RESULTS_DIR / "ablation_a5_det_ode.csv", index=False)


## Ablation summary table

In [ ]:
# Load SolarSDE full (A1) from Notebook 2
a1 = pd.read_csv(RESULTS_DIR / "solar_sde_main_results.csv").copy()
a1["variant"] = "A1_full"

abl = pd.concat([a1, df_a2, df_a4, df_a5], ignore_index=True)
abl = abl[["variant", "horizon_min", "crps", "rmse", "mae", "picp", "pinaw", "ramp_crps"]]
abl.to_csv(RESULTS_DIR / "ablation_results.csv", index=False)

print("=" * 80)
print("ABLATION RESULTS")
print("=" * 80)
print(abl.to_string(index=False))

# Relative change vs A1 (full) at 10-min horizon
a1_crps_10 = float(abl[(abl["variant"]=="A1_full") & (abl["horizon_min"]==10)]["crps"].iloc[0])
print(f"\nRelative CRPS change vs A1 (full) at 10-min horizon:")
for v in ["A2_no_cti", "A4_no_score", "A5_deterministic_ode"]:
    row = abl[(abl["variant"]==v) & (abl["horizon_min"]==10)]
    if not row.empty:
        crps_v = float(row["crps"].iloc[0])
        delta = (crps_v - a1_crps_10) / a1_crps_10 * 100
        print(f"  {v}: CRPS={crps_v:.3f}  (Δ = {delta:+.1f}%)")


In [ ]:
# ==== Zip outputs and prepare download ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} -> {zip_path}")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive size: {size_mb:.1f} MB")

if IN_COLAB:
    from google.colab import files
    try:
        files.download(str(zip_path))
        print("Download triggered (check browser).")
    except Exception as e:
        print(f"Auto-download unavailable: {e}. Download manually from {zip_path}")
else:
    print(f"Archive at: {zip_path}  — download via Kaggle output tab or file browser.")


In [ ]:
print("=" * 70); print("NOTEBOOK 4 COMPLETE"); print("=" * 70)
print("Next: 05_analysis_figures.ipynb")
